In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,PowerTransformer,RobustScaler,OneHotEncoder,OrdinalEncoder
from sklearn.metrics import accuracy_score,recall_score,f1_score,precision_score,roc_auc_score,confusion_matrix,classification_report
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from collections import Counter
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input,BatchNormalization
from sklearn.utils.class_weight import compute_class_weight
import optuna
from tensorflow.keras.optimizers import RMSprop,SGD,Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
df = pd.read_csv(r"/kaggle/input/datasets/santhoshkumarpuppala/american-bankruptcy-data/american_bankruptcy_dataset.csv")

In [ ]:
df.rename(columns={
    "X1": "Current_Assets",
    "X2": "Cost_of_Goods_Sold",
    "X3": "Depreciation_and_Amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net_Income",
    "X7": "Total_Receivables",
    "X8": "Market_Value",
    "X9": "Net_Sales",
    "X10": "Total_Assets",
    "X11": "Long_Term_Debt",
    "X12": "EBIT",
    "X13": "Gross_Profit",
    "X14": "Current_Liabilities",
    "X15": "Retained_Earnings",
    "X16": "Total_Revenue",
    "X17": "Total_Liabilities",
    "X18": "Operating_Expenses"}, inplace=True)

In [ ]:
df.drop(columns=["Total_Liabilities"], inplace=True)

In [ ]:
train = df[df["fyear"].between(1999, 2014)]
test = df[df["fyear"].between(2015, 2018)]

print("Training:", train.shape)
print("Testing:", test.shape)

In [ ]:
X_train = train.drop(columns = ['company_name','status_label'])
y_train = train['status_label']
X_test = test.drop(columns = ['company_name','status_label'])
y_result = test['status_label']

In [ ]:
numerical_pipeline = Pipeline([("Scaler", RobustScaler())])

In [ ]:
numerical_features = ["Current_Assets","Cost_of_Goods_Sold","Depreciation_and_Amortization","EBITDA","Inventory","Net_Income","Total_Receivables","Market_Value","Net_Sales","Total_Assets","Long_Term_Debt",
    "EBIT","Gross_Profit","Retained_Earnings","Total_Revenue","Current_Liabilities","Operating_Expenses"]

categorical_features = ["MajorGroup",'Division']
ordina_columns = ['fyear']

In [ ]:
categorical_pipeline = Pipeline([("Encoder", OneHotEncoder(handle_unknown="ignore",sparse_output=True))])

In [ ]:
preprocessor = ColumnTransformer([("Numerical", numerical_pipeline, numerical_features),("Categorical", categorical_pipeline, categorical_features),('ordinal',)])

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
classes = np.unique(y_train)

weights = compute_class_weight(class_weight="balanced",classes=classes,y=y_train)

class_weights = dict(zip(classes, weights))

print(class_weights)

In [ ]:
label_map = {'alive': 0, 'failed': 1}
y_train = train['status_label'].map(label_map)

y_result = test['status_label'].map(label_map)

In [77]:
def build_baseline_model(input_shape, units):

    model = Sequential()

    model.add(Input(shape=(input_shape,)))

    # Only one hidden layer
    model.add(Dense(units, activation="relu"))

    # Output layer
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.F1Score(threshold=0.5, name="f1_score")
        ]
    )

    return model

In [78]:
model_baseline = build_baseline_model(
    X_train_processed.shape[1]
,128)

history_baseline = model_baseline.fit(
    X_train_processed,
    y_train,
    validation_data=(X_tr, y_val),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)

Epoch 1/50
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - f1_score: 0.0326 - loss: 0.5046 - precision: 0.1941 - recall: 0.0178 - val_f1_score: 0.0712 - val_loss: 0.2399 - val_precision: 0.4419 - val_recall: 0.0387
Epoch 2/50
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.0427 - loss: 0.4018 - precision: 0.2655 - recall: 0.0232 - val_f1_score: 0.1791 - val_loss: 0.3223 - val_precision: 0.3211 - val_recall: 0.1242
Epoch 3/50
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.0529 - loss: 0.4119 - precision: 0.3208 - recall: 0.0288 - val_f1_score: 0.1700 - val_loss: 0.2209 - val_precision: 0.4679 - val_recall: 0.1039
Epoch 4/50
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.0600 - loss: 0.7220 - precision: 0.3468 - recall: 0.0329 - val_f1_score: 0.0080 - val_loss: 0.2356 - val_precision: 0.2222 - val_recall: 0.0041
Epoch 5/50
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - f1_score: 0.0710 - loss: 0.4723 - precision: 0.3766 - recall: 0.0392 - val_f1_score: 0.2266 - v

In [79]:
y_val_prob = model_baseline.predict(X_val_processed).ravel()
y_val_pred = (y_val_prob >= 0.5).astype(int)
print(classification_report(y_val,y_val_pred,target_names=["Alive", "Failed"]))
print(confusion_matrix(y_val, y_val_pred))

328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
              precision    recall  f1-score   support

       Alive       0.96      0.97      0.97      9982
      Failed       0.30      0.23      0.26       491

    accuracy                           0.94     10473
   macro avg       0.63      0.60      0.61     10473
weighted avg       0.93      0.94      0.93     10473

[[9715  267]
 [ 377  114]]


In [80]:
model_final_batch = Sequential()

#inout layer
model_final_batch.add(Input(shape = (X_train_processed.shape[1],)))

#hidden layers
model_final_batch.add(Dense(units = 128,activation='relu'))
model_final_batch.add(BatchNormalization())
model_final_batch.add(Dropout (0.3))

model_final_batch.add(Dense(units = 32,activation='tanh'))
model_final_batch.add(BatchNormalization())
model_final_batch.add(Dropout (0.2))

model_final_batch.add(Dense(units = 128,activation='tanh'))
model_final_batch.add(BatchNormalization())
model_final_batch.add(Dropout (0.1))

model_final_batch.add(Dense(units = 64,activation='relu'))
model_final_batch.add(BatchNormalization())
model_final_batch.add(Dropout (0.2))


#output layer
model_final_batch.add(Dense(units = 1 ,activation = 'sigmoid'))

# Optimizer
optimizer = Adam(learning_rate = 0.0006622442381609686)

# Compile
model_final_batch.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.F1Score(threshold = 0.7,name="f1_score")])

In [81]:
early_stop = EarlyStopping(monitor="val_f1_score",mode="max",patience=10,
    restore_best_weights=True)

history = model_final_batch.fit(X_train_processed,y_train,
                    validation_data=(X_val_processed,y_val),
                    epochs=100,
                    batch_size=128,
                    class_weight=class_weights,
                    callbacks=[early_stop],
                    verbose=1)

Epoch 1/100
437/437 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.8243 - f1_score: 0.0401 - loss: 0.4440 - precision: 0.0804 - recall: 0.1162 - val_accuracy: 0.9531 - val_f1_score: 0.0000e+00 - val_loss: 0.1942 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/100
437/437 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9172 - f1_score: 0.0000e+00 - loss: 0.2935 - precision: 0.1157 - recall: 0.0063 - val_accuracy: 0.9531 - val_f1_score: 0.0000e+00 - val_loss: 0.1821 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
437/437 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9193 - f1_score: 4.4964e-04 - loss: 0.2824 - precision: 0.1224 - recall: 0.0027 - val_accuracy: 0.9531 - val_f1_score: 0.0000e+00 - val_loss: 0.1803 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/100
437/437 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9197 - f1_score: 0.0000e+00 - loss: 0.2756 - precision: 0.0385 - recall: 4.5025e-04 - val_accuracy: 0.9531 - val_f1_score: 0.00

In [82]:
print("final_model_batch")

y_probability = model_final_batch.predict(X_test_processed).ravel()
y_pred = (y_probability >= 0.7).astype(int)

print(classification_report(y_result, y_pred, target_names=["Alive", "Failed"]))
print("Confusion Matrix:")
print(confusion_matrix(y_result, y_pred))
print("Precision: ", precision_score(y_result, y_pred))
print("Recall: ", recall_score(y_result, y_pred))
print("F1_score: ", f1_score(y_result, y_pred))


final_model_batch
384/384 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
              precision    recall  f1-score   support

       Alive       0.98      1.00      0.99     11995
      Failed       0.00      0.00      0.00       287

    accuracy                           0.98     12282
   macro avg       0.49      0.50      0.49     12282
weighted avg       0.95      0.98      0.97     12282

Confusion Matrix:
[[11995     0]
 [  287     0]]
Precision:  0.0
Recall:  0.0
F1_score:  0.0


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/m

In [72]:

# Features where negative values need investigation

investigate_features = [
    'Current_Assets',
    'Total_Assets',
    'Current_Liabilities',
    'Long_Term_Debt',
    'Inventory',
    'Total_Receivables',
    'Operating_Expenses'
]

negative_summary = pd.DataFrame({
    'Negative_Count': (df[investigate_features] < 0).sum(),
    'Negative_Percentage': (
        (df[investigate_features] < 0).mean() * 100
    ).round(3),
    'Minimum_Value': df[investigate_features].min()
})

print(negative_summary)


                     Negative_Count  Negative_Percentage  Minimum_Value
Current_Assets                    6                0.008  -2.498000e+03
Total_Assets                      0                0.000   7.000000e-03
Current_Liabilities              13                0.017  -1.964999e+06
Long_Term_Debt                34001               43.213  -1.605370e+07
Inventory                         0                0.000   0.000000e+00
Total_Receivables             23012               29.247  -4.512135e+06
Operating_Expenses                3                0.004  -3.171970e+05


In [73]:

suspicious_features = [
    'Current_Assets',
    'Current_Liabilities',
    'Long_Term_Debt',
    'Total_Receivables',
    'Operating_Expenses'
]

for feature in suspicious_features:

    print("\n" + "=" * 70)
    print("FEATURE:", feature)
    print("=" * 70)

    negative_rows = df[df[feature] < 0]

    columns_to_display = [
        'fyear',
        'status_label',
        'MajorGroup',
        'Current_Assets',
        'Total_Assets',
        'Current_Liabilities',
        'Long_Term_Debt',
        'Total_Receivables',
        'Operating_Expenses'
    ]

    print(
        negative_rows[columns_to_display]
        .head(10)
        .to_string(index=False)
    )



FEATURE: Current_Assets
                        fyear status_label  MajorGroup  Current_Assets  Total_Assets  Current_Liabilities  Long_Term_Debt  Total_Receivables  Operating_Expenses
1970-01-01 00:00:00.000002002        alive          58          -13.00        428.00              248.000         -51.000             -27.00              275.00
1970-01-01 00:00:00.000002008        alive          48          -11.00        635.00                0.227       -3108.000           -1938.00             2165.00
1970-01-01 00:00:00.000002008        alive          10           -1.00       2739.00              629.000       -4603.000           -3666.00             4295.00
1970-01-01 00:00:00.000002015        alive           1        -1124.00        596.00              108.000          -1.583              -0.72              828.00
1970-01-01 00:00:00.000002016        alive           1           -7.76          0.61                0.470       -6809.000              -3.14                3.61
1970-01-0

In [74]:
 
for feature in suspicious_features:

    negative_by_year = (
        df[df[feature] < 0]
        .groupby('fyear')
        .size()
    )

    print(f"\nNegative values in {feature}:")
    print(negative_by_year)



Negative values in Current_Assets:
fyear
1970-01-01 00:00:00.000002002    1
1970-01-01 00:00:00.000002008    2
1970-01-01 00:00:00.000002015    1
1970-01-01 00:00:00.000002016    1
1970-01-01 00:00:00.000002017    1
dtype: int64

Negative values in Current_Liabilities:
fyear
1970-01-01 00:00:00.000002003    1
1970-01-01 00:00:00.000002005    1
1970-01-01 00:00:00.000002008    2
1970-01-01 00:00:00.000002009    2
1970-01-01 00:00:00.000002010    1
1970-01-01 00:00:00.000002011    3
1970-01-01 00:00:00.000002013    1
1970-01-01 00:00:00.000002014    1
1970-01-01 00:00:00.000002015    1
dtype: int64

Negative values in Long_Term_Debt:
fyear
1970-01-01 00:00:00.000001999    2257
1970-01-01 00:00:00.000002000    2485
1970-01-01 00:00:00.000002001    2617
1970-01-01 00:00:00.000002002    2380
1970-01-01 00:00:00.000002003    1933
1970-01-01 00:00:00.000002004    1635
1970-01-01 00:00:00.000002005    1569
1970-01-01 00:00:00.000002006    1507
1970-01-01 00:00:00.000002007    1556
1970-01-01 

In [75]:
print("Before correction:")
print(df['fyear'].dtype)
print(df['fyear'].head(10))

# Recover the original integer values if they were converted
# directly from integers using pd.to_datetime()
df['fyear'] = df['fyear'].astype('int64')

print("\nAfter correction:")
print(df['fyear'].dtype)
print(df['fyear'].head(10))

print("\nYear range:")
print(df['fyear'].min(), df['fyear'].max())

Before correction:
datetime64[ns]
0   1970-01-01 00:00:00.000001999
1   1970-01-01 00:00:00.000002000
2   1970-01-01 00:00:00.000002001
3   1970-01-01 00:00:00.000002002
4   1970-01-01 00:00:00.000002003
5   1970-01-01 00:00:00.000002004
6   1970-01-01 00:00:00.000002005
7   1970-01-01 00:00:00.000002006
8   1970-01-01 00:00:00.000002007
9   1970-01-01 00:00:00.000002008
Name: fyear, dtype: datetime64[ns]

After correction:
int64
0    1999
1    2000
2    2001
3    2002
4    2003
5    2004
6    2005
7    2006
8    2007
9    2008
Name: fyear, dtype: int64

Year range:
1999 2018


In [76]:
for feature in [
    'Current_Assets',
    'Current_Liabilities',
    'Long_Term_Debt',
    'Total_Receivables',
    'Operating_Expenses'
]:

    print(f"\nNegative values in {feature}:")

    print(
        df.loc[df[feature] < 0]
        .groupby('fyear')
        .size()
    )


Negative values in Current_Assets:
fyear
2002    1
2008    2
2015    1
2016    1
2017    1
dtype: int64

Negative values in Current_Liabilities:
fyear
2003    1
2005    1
2008    2
2009    2
2010    1
2011    3
2013    1
2014    1
2015    1
dtype: int64

Negative values in Long_Term_Debt:
fyear
1999    2257
2000    2485
2001    2617
2002    2380
2003    1933
2004    1635
2005    1569
2006    1507
2007    1556
2008    1801
2009    1702
2010    1320
2011    1263
2012    1402
2013    1493
2014    1534
2015    1565
2016    1480
2017    1344
2018    1158
dtype: int64

Negative values in Total_Receivables:
fyear
1999    1543
2000    1769
2001    1727
2002    1490
2003    1250
2004    1122
2005    1082
2006    1112
2007    1111
2008    1094
2009    1065
2010     880
2011     866
2012     965
2013    1045
2014    1091
2015    1103
2016    1019
2017     900
2018     778
dtype: int64

Negative values in Operating_Expenses:
fyear
2000    1
2009    1
2014    1
dtype: int64


In [17]:
features = [
    'Long_Term_Debt',
    'Total_Receivables'
]

for feature in features:

    print("\n", "=" * 60)
    print("FEATURE:", feature)
    print("=" * 60)

    negative_records = df[df[feature] < 0]

    print(
        negative_records[
            [
                'fyear',
                'status_label',
                'MajorGroup',
                'Total_Assets',
                'Current_Liabilities',
                'Long_Term_Debt',
                'Total_Receivables'
            ]
        ].head(20).to_string(index=False)
    )


FEATURE: Long_Term_Debt
 fyear status_label  MajorGroup  Total_Assets  Current_Liabilities  Long_Term_Debt  Total_Receivables
2001.0        alive          37      150464.0            638721.00         -58.939            27207.0
2002.0        alive          37      203575.0            606337.00         -12.410            30745.0
2001.0        alive          36         599.4              2402.80       -1287.700              -55.6
2002.0        alive          36         397.8              1047.70       -1145.000             -192.4
2003.0        alive          36         265.5               773.20         -76.700               19.9
2008.0        alive          36         278.0              1456.40         -41.900              173.3
2009.0        alive          36         236.0               996.70        -474.300               84.7
1999.0        alive          38        2804.0                29.37       -2207.000             -265.0
2000.0        alive          38        2278.0            

In [18]:
feature_mapping = {
    'X7': 'Total_Receivables',
    'X11': 'Long_Term_Debt'
}

In [21]:
df1 = pd.read_csv(r"/kaggle/input/datasets/santhoshkumarpuppala/american-bankruptcy-data/american_bankruptcy_dataset.csv")

In [23]:
print(
    df1[
        ['X7', 'X11', 'status_label', 'fyear']
    ].describe()
)

print("\nNegative X7:", (df1['X7'] < 0).sum())
print("Negative X11:", (df1['X11'] < 0).sum())

                 X7           X11         fyear
count  7.868200e+04  7.868200e+04  78682.000000
mean   6.942115e+04  1.397351e+04   2007.506317
std    2.847994e+05  1.978544e+05      5.742768
min   -4.512135e+06 -1.605370e+07   1999.000000
25%   -9.000000e+00 -2.636000e+03   2002.000000
50%    1.121900e+03  3.137400e+01   2007.000000
75%    3.325675e+04  8.379750e+03   2012.000000
max    1.256672e+07  7.521731e+06   2018.000000

Negative X7: 23012
Negative X11: 34001
